> ## SOLUTION / ANSWER KEY &mdash; Lab 8.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-customer-service-chatbot.ipynb`](../lab-12-capstone-customer-service-chatbot.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.12 &mdash; Capstone: The Multi-Agent Chatbot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Chain route -> specialists -> synthesise into one handler
- Gate any irreversible action (a refund) on human approval
- Run the chatbot over a SUITE of messages and score it

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The multi-agent customer-service chatbot](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-12"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Capstone: the **multi-agent customer-service chatbot** (the client's Lab 4.2), end to end. A
**supervisor** routes each message to the matching **specialists**; each returns a grounded finding; a
**synthesiser** composes one reply; and anything **irreversible** (a refund) is flagged
**`needs_approval`** for a human &mdash; never auto-done. You run it over a **suite** of messages and
score the outcomes. The pieces below are the ones you built through the module; you assemble them.

In [ ]:


# The pieces you built this module, provided here so you can assemble the whole chatbot.
SPECIALISTS = {
    "billing": ["charge", "refund", "invoice", "billed"],
    "tech":    ["crash", "bug", "login", "broken"],
}
def route(message):
    m = message.lower()
    hits = [name for name, kws in SPECIALISTS.items() if any(k in m for k in kws)]
    return hits if hits else ["general"]

def billing_agent(message):  return "On billing: duplicate charge -> refund warranted."
def tech_agent(message):     return "On the app: matches BUG-231 -> update to v4.2."
def general_agent(message):  return "On your query: forwarded to a human agent."
AGENTS = {"billing": billing_agent, "tech": tech_agent, "general": general_agent}

def synthesize(findings):
    return " ".join(findings[k] for k in sorted(findings)) if findings else "No findings."
print("route, agents & synthesise ready")

## Your Turn
Assemble `process` (route -> run each specialist -> synthesise -> gate the refund) and `evaluate`.

In [ ]:
def process(message):
    involved = route(message)
    findings = {name: AGENTS[name](message) for name in involved}
    reply = synthesize(findings)
    # irreversible action? a refund needs a human before anything happens
    needs_human = any("refund" in f.lower() for f in findings.values())
    status = "needs_approval" if needs_human else "auto_ok"
    return {"agents": sorted(findings), "reply": reply, "status": status}

SUITE = [
    {"message": "charged twice for 4471 and the app keeps crashing", "agents": ["billing", "tech"], "status": "needs_approval"},
    {"message": "the app keeps crashing on login",                  "agents": ["tech"],            "status": "auto_ok"},
    {"message": "please refund my invoice",                         "agents": ["billing"],         "status": "needs_approval"},
]

def evaluate():
    solved = sum(1 for t in SUITE if process(t["message"])["agents"] == t["agents"] and process(t["message"])["status"] == t["status"])
    return solved, len(SUITE)

try:
    for t in SUITE:
        r = process(t['message'])
        print(r['agents'], '|', r['status'], '->', r['reply'][:48])
    print('score:', evaluate())
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a two-intent message engages both specialists", lambda: process("charged twice and the app keeps crashing")["agents"] == ["billing", "tech"])
expect_true("a pure tech message engages only tech", lambda: process("the app keeps crashing")["agents"] == ["tech"])
expect_true("the reply is synthesised from the findings", lambda: (lambda r: "refund" in r.lower() and "bug-231" in r.lower())(process("charged twice and crashing")["reply"]))
expect_true("a refund is gated on human approval", lambda: process("please refund my invoice")["status"] == "needs_approval")
expect_true("a non-irreversible case is auto_ok", lambda: process("the app keeps crashing")["status"] == "auto_ok")
expect_true("the chatbot solves the whole suite", lambda: evaluate() == (3, 3))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
See a REAL model do the supervisor's intent-routing (Ollama / Groq) -- the bridge to Day 5. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        from langchain_ollama import ChatOllama
        llm = ChatOllama(model="llama3.2:1b")
        msg = "I was charged twice and the app keeps crashing."
        intents = llm.invoke("List the intents (billing/tech) in one line: " + msg).content
        print("REAL supervisor intents:", intents)
        print("\nProduction shape (LangGraph): supervisor node routes to billing & tech agent nodes,")
        print("findings flow through shared state to a synthesise node, and the refund waits on human approval.")
    else:
        print("No Ollama reachable -- skipping the live routing. The offline multi-agent chatbot above already ran")
        print("the whole suite -- routed, synthesised, and gated every refund on human approval.")
    print("Next: Day 5 -- agents in industry (finance / health / cyber) and responsible AI.")
except Exception as e:
    print("Live routing skipped:", type(e).__name__)

---
### Key takeaway
You built a multi-agent customer-service chatbot end to end -- specialists coordinated by a supervisor, findings synthesised into one reply, refunds gated on a human. That completes Day 4. Next: Day 5 -- agents in the real world, responsibly.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>